# Extract parish ministries

Predict ministries from the page information generated by `02_build_page_list.ipynb`. Inputs are read from `data/02_build_page_list/`; prediction outputs are saved in `data/03_extract_ministries/`. Run this notebook from the project root.


## 0 Environment setup
Install project dependencies with `%pip install -e .` if needed. Select `LLM_PROVIDER=ollama` or `openai` (alias `chatgpt`) in `.env`. Set `LLM_MODEL` to a model for that provider and `LLM_CACHE_DIR` to the cache folder. OpenAI uses `OPENAI_API_KEY`; Ollama uses the local server (or `OLLAMA_HOST`). Existing environment variables take precedence.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from utils.llm import LLMClient
from utils.ministry_prediction import EXTRACTOR_VERSION, predict_ministry
from utils.ministry_cache import (
    extractor_cache_dir, load_cached_pages, page_cache_key,
    page_input_hash, write_json_atomic, write_csv_atomic,
)

_ = load_dotenv(".env", override=False)


## 1 Predict at most one ministry per page
Use `utils.llm.LLMClient` with the provider/model configured in `.env`, structured JSON output, Pydantic validation, and temperature 0. See `.env.example` for the configuration settings. Each request receives parish details and all available information for one page: its URL, discovery sources, title, text, and other supplied page fields. No page content is downloaded here; URL-only predictions remain provisional.

`EXTRACTOR_VERSION` in `utils/ministry_prediction.py` is now `2.0.0` for this page-level extractor. Increment it when changing the prompt, categories, or response semantics. Provider, version, and model isolate the cache; changed page/parish information invalidates that page's cached prediction. Reordering rows or changing `parish_index` does not invalidate results. Previous parish-level caches and JSON outputs are retained but are not reused for this different extraction task.

Set `PAGE_LIMIT` to the maximum number of **new LLM calls** in this run, or `None` for all pending pages. Cached pages do not count toward the limit: repeated limited runs advance to the next pending pages in discovery order. Failed requests count toward the limit and retry on subsequent runs.

Read `data/02_build_page_list/1_site_pages.csv` and enrich each page from `2_page_content.json`, matching `site_url` and `page_url` when available (older URL-only files are supported). Cache keys include parish identity and page URL, so parishes sharing a page URL remain separate.

Write `data/03_extract_ministries/parish_ministries.csv`, with one row per processed parish/page pair, combining successful cached rows for the current version/model with this run's results. Include parish and page fields, separate scalar `ministry` and `category` columns, provider (`llm_provider`), model, version, input hash, and prediction status/error. A page has at most one ministry and exactly one category for that ministry. If no single primary ministry is evident, both columns are empty with status `ok`; errors also leave both columns empty but include status `error` and a diagnostic message.

Successful responses (including pages with no ministry) are saved immediately as one JSON file per parish/page pair under `LLM_CACHE_DIR/ministry_extraction/`. The combined CSV is checkpointed after every request and at the end. Interrupted runs retain completed cache entries. Progress starts with `cache` or `llm`. The final cell lists all distinct ministry/category pairs and categories across the combined CSV.

The shared module also caches validated LLM responses under `LLM_CACHE_DIR/responses/`, keyed by provider, model, messages, schema, temperature, and extractor version. Existing compatible page checkpoints are copied from the previous cache location on first use, preserving processed pages. The CSV keeps its existing columns and adds `llm_provider` so the selected service is recorded.


In [ ]:
llm = LLMClient()  # Provider, model, and cache folder come from .env.
llm_model = llm.model
PAGE_LIMIT = 10  # Maximum new LLM calls this run; None for all pending pages.

if PAGE_LIMIT is not None and (type(PAGE_LIMIT) is not int or PAGE_LIMIT < 0):
    raise ValueError("PAGE_LIMIT must be None or a nonnegative integer")

input_dir = Path("data/02_build_page_list")
output_dir = Path("data/03_extract_ministries")
site_pages_df = pd.read_csv(input_dir / "1_site_pages.csv", keep_default_na=False)
selected_pages = site_pages_df.copy()
with (input_dir / "2_page_content.json").open(encoding="utf-8") as file:
    page_content_records = json.load(file)
content_df = pd.DataFrame(page_content_records)
join_keys = ["site_url", "page_url"] if "site_url" in content_df.columns else ["page_url"]
parish_columns = ["parish_index", "name", "diocese", "site_url"]
# Parish identity comes from discovery; retain all other available page information.
content_columns = join_keys + [
    column for column in content_df.columns
    if column not in join_keys and column not in parish_columns
]
content_df = content_df.reindex(columns=content_columns).fillna("").drop_duplicates()
for key in join_keys:
    content_df[key] = content_df[key].astype("string")
    selected_pages[key] = selected_pages[key].astype("string")
# Content fields override the same fields from discovery, without multiplying page rows.
overlap = [column for column in content_columns if column in selected_pages.columns and column not in join_keys]
selected_pages = selected_pages.drop(columns=overlap).merge(
    content_df, on=join_keys, how="left", validate="many_to_one",
).fillna("")
cache_dir = extractor_cache_dir(
    output_dir, EXTRACTOR_VERSION, llm_model,
    provider=llm.provider, cache_root=llm.cache_dir,
)
output_path = output_dir / "parish_ministries.csv"
cached_results = load_cached_pages(cache_dir, EXTRACTOR_VERSION, llm_model, provider=llm.provider)
# Reuse existing page checkpoints after moving the cache setting to .env.
previous_cache_dir = extractor_cache_dir(output_dir, EXTRACTOR_VERSION, llm_model, provider=llm.provider)
if previous_cache_dir != cache_dir:
    for key, record in load_cached_pages(previous_cache_dir, EXTRACTOR_VERSION, llm_model, provider=llm.provider).items():
        if key not in cached_results:
            record["llm_provider"] = llm.provider
            write_json_atomic(cache_dir / f"{key}.json", record)
            cached_results[key] = record
combined_results = dict(cached_results)
result_columns = [
    "ministry", "category", "llm_provider", "llm_model", "extractor_version", "input_hash",
    "prediction_status", "prediction_error",
]
# Include additional fields present in earlier cached batches as well as current inputs.
output_columns = list(dict.fromkeys(
    list(selected_pages.columns)
    + [column for row in cached_results.values() for column in row if column not in result_columns]
    + result_columns
))
new_calls = 0
cache_hits = 0
print(f"Extractor {EXTRACTOR_VERSION}; provider {llm.provider}; model {llm_model}; {len(cached_results)} cached pages")

for page in selected_pages.to_dict("records"):
    key = page_cache_key(page)
    input_hash = page_input_hash(page)
    cached = cached_results.get(key)
    if cached is not None and cached["input_hash"] == input_hash:
        combined_results[key] = {**cached, **page, "llm_provider": llm.provider}
        cache_hits += 1
        print(f"cache {page['name']} {page['page_url']}: {cached['ministry'] or 'no ministry'}")
        continue
    if PAGE_LIMIT is not None and new_calls >= PAGE_LIMIT:
        continue

    new_calls += 1
    ministry = ""
    category = ""
    status = "ok"
    error = ""
    try:
        result = predict_ministry(page, llm=llm)
        if result.ministry is not None:
            ministry = result.ministry.name
            category = result.ministry.category
    except Exception as exc:
        # Errors stay eligible for retry; successful no-ministry results are cached.
        status = "error"
        error = f"{type(exc).__name__}: {exc}"
    record = {
        **page,
        "ministry": ministry,
        "category": category,
        "llm_provider": llm.provider,
        "llm_model": llm_model,
        "extractor_version": EXTRACTOR_VERSION,
        "input_hash": input_hash,
        "prediction_status": status,
        "prediction_error": error,
    }
    if status == "ok":
        write_json_atomic(cache_dir / f"{key}.json", record)
        cached_results[key] = record
    combined_results[key] = record
    write_csv_atomic(output_path, pd.DataFrame(combined_results.values(), columns=output_columns))
    print(f"llm {new_calls} {page['name']} {page['page_url']}: {ministry or 'no ministry'} ({status})")

page_results_df = pd.DataFrame(combined_results.values(), columns=output_columns)
write_csv_atomic(output_path, page_results_df)
print(f"Saved {len(page_results_df)} combined page results to {output_path}; {cache_hits} cache hits, {new_calls} LLM calls")
page_results_df.head()


In [ ]:
# Inspect ministries/categories across cached and newly processed pages.
ministries_df = page_results_df.loc[
    (page_results_df["prediction_status"] == "ok") & (page_results_df["ministry"] != ""),
    ["ministry", "category"],
]
ministries_categories_df = (
    ministries_df.groupby(["ministry", "category"]).size().reset_index(name="page_count")
    .sort_values(["category", "ministry"]).reset_index(drop=True)
)
ministry_categories_df = ministries_df[["category"]].drop_duplicates().sort_values("category").reset_index(drop=True)
display(ministries_categories_df)
display(ministry_categories_df)
